# GLB Pipeline: Photorealistic 3D Tiles → Meshopt-compressed GLB

This notebook walks through both halves of how `Google_LasVegas_Export_v32.glb` was built:

| Part | What it does |
|---|---|
| **1 – 3D Tiles API** | Fetch Google's live photorealistic tile stream, walk the tile tree, download raw GLB tiles for a bounding area |
| **2 – Meshopt** | Load a GLB, apply `EXT_meshopt_compression`, compare file sizes |

> **Note**: Part 1 requires a Google Maps Platform API key with the *Map Tiles API* enabled.  
> Part 2 works fully offline on any `.glb` file.

## Setup — install dependencies

In [ ]:
%pip install requests pygltflib meshoptimizer numpy --quiet

In [ ]:
import os
import json
import struct
import hashlib
import pathlib
import requests
import numpy as np
import pygltflib
import meshoptimizer

# Where to save downloaded tiles
TILE_CACHE = pathlib.Path("tile_cache")
TILE_CACHE.mkdir(exist_ok=True)

print("pygltflib version:", pygltflib.__version__)
print("meshoptimizer version:", meshoptimizer.__version__)

---
## Part 1 — Google Photorealistic 3D Tiles API

### How it works

```
tile.googleapis.com/v1/3dtiles/root.json
        │
        └── tileset JSON  (OGC 3D Tiles spec)
                │
                ├── root tile  { boundingVolume, geometricError, children[] }
                │       ├── child tile  →  content.uri  →  .b3dm / .glb
                │       └── child tile  →  content.uri  →  another tileset.json
                └── ...
```

Each leaf tile's `content.uri` points to a binary glTF (`.glb` or `.b3dm`) that contains
the mesh + photorealistic texture for that geographic region.

### 1.1 — Configuration

In [ ]:
# Set your API key here, or via environment variable GOOGLE_MAPS_API_KEY
API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_API_KEY_HERE")

TILES_ROOT = "https://tile.googleapis.com/v1/3dtiles/root.json"

# Las Vegas Strip bounding box (lon_min, lat_min, lon_max, lat_max)
# Matches the GLB_CENTER used in road-network.ts: lon=-115.1769, lat=36.1085
LV_BBOX = {
    "lon_min": -115.20,
    "lat_min":  36.08,
    "lon_max": -115.14,
    "lat_max":  36.14,
}

# Maximum tiles to download in one run (cost + time guard)
MAX_TILES = 5

if API_KEY == "YOUR_API_KEY_HERE":
    print("⚠  No API key set — tile fetching will fail. Set GOOGLE_MAPS_API_KEY.")
else:
    print(f"✓  API key loaded ({API_KEY[:8]}…)")

### 1.2 — Fetch the root tileset

In [ ]:
def fetch_json(url: str, params: dict = None) -> dict:
    """GET a JSON resource, appending the API key."""
    p = {"key": API_KEY, **(params or {})}
    resp = requests.get(url, params=p, timeout=15)
    resp.raise_for_status()
    return resp.json()


root_tileset = fetch_json(TILES_ROOT)

print("Root tileset keys:", list(root_tileset.keys()))
print("Geometric error:", root_tileset["root"].get("geometricError"))
print("Children:", len(root_tileset["root"].get("children", [])))

### 1.3 — Walk the tile tree, find tiles in the Las Vegas bounding box

In [ ]:
import math
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse


def bv_center_lon_lat(bv: dict) -> tuple[float, float] | None:
    """Extract (lon, lat) centre from a boundingVolume.

    3D Tiles bounding volumes come in three flavours:
      - region  [west, south, east, north, minH, maxH]  (radians)
      - box     [cx,cy,cz, + 9 half-axis values]        (ECEF metres)
      - sphere  [cx,cy,cz, radius]                      (ECEF metres)
    """
    if "region" in bv:
        west, south, east, north = bv["region"][:4]
        lon = math.degrees((west + east) / 2)
        lat = math.degrees((south + north) / 2)
        return lon, lat

    if "box" in bv:
        # ECEF centre → lon/lat
        cx, cy, cz = bv["box"][:3]
        lat = math.degrees(math.atan2(cz, math.sqrt(cx**2 + cy**2)))
        lon = math.degrees(math.atan2(cy, cx))
        return lon, lat

    if "sphere" in bv:
        cx, cy, cz = bv["sphere"][:3]
        lat = math.degrees(math.atan2(cz, math.sqrt(cx**2 + cy**2)))
        lon = math.degrees(math.atan2(cy, cx))
        return lon, lat

    return None


def in_bbox(lon: float, lat: float, bbox: dict) -> bool:
    return (
        bbox["lon_min"] <= lon <= bbox["lon_max"]
        and bbox["lat_min"] <= lat <= bbox["lat_max"]
    )


def append_key(url: str) -> str:
    """Append the API key to a tile URL if not already present."""
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    qs["key"] = [API_KEY]
    new_query = urlencode({k: v[0] for k, v in qs.items()})
    return urlunparse(parsed._replace(query=new_query))


def resolve_uri(base_url: str, uri: str) -> str:
    """Resolve a possibly-relative tile URI against its parent tileset URL."""
    if uri.startswith("http"):
        return append_key(uri)
    # Relative — strip query from base, join, re-add key
    base_no_qs = urlunparse(urlparse(base_url)._replace(query=""))
    return append_key(urljoin(base_no_qs + "/", uri))


glb_urls: list[str] = []


def walk_tile(tile: dict, tileset_url: str, depth: int = 0) -> None:
    """Recursively walk a tile node, collecting content URLs in the Las Vegas bbox."""
    if len(glb_urls) >= MAX_TILES:
        return

    bv = tile.get("boundingVolume", {})
    centre = bv_center_lon_lat(bv)

    # Prune branches that are clearly outside the bounding box
    if centre and depth > 1 and not in_bbox(*centre, LV_BBOX):
        return

    content = tile.get("content") or tile.get("contents", [{}])[0]
    uri = content.get("uri") or content.get("url", "")

    if uri.endswith(".json"):
        # Nested tileset — fetch and recurse
        try:
            nested_url = resolve_uri(tileset_url, uri)
            nested = requests.get(nested_url, timeout=15).json()
            walk_tile(nested["root"], nested_url, depth + 1)
        except Exception as exc:
            print(f"  [skip nested] {exc}")

    elif uri.endswith((".glb", ".b3dm")) and centre and in_bbox(*centre, LV_BBOX):
        full_url = resolve_uri(tileset_url, uri)
        print(f"  {'  ' * depth}✓ tile lon={centre[0]:.4f} lat={centre[1]:.4f}  {uri[-40:]}")
        glb_urls.append(full_url)

    for child in tile.get("children", []):
        walk_tile(child, tileset_url, depth + 1)


print(f"Walking tile tree — collecting up to {MAX_TILES} tiles in Las Vegas bbox…\n")
walk_tile(root_tileset["root"], TILES_ROOT)
print(f"\nFound {len(glb_urls)} tile(s)")

### 1.4 — Download the tiles

In [ ]:
def download_tile(url: str) -> pathlib.Path:
    """Download a tile to TILE_CACHE, keyed by URL hash."""
    key = hashlib.md5(url.encode()).hexdigest()[:10]
    ext = ".glb" if ".glb" in url else ".b3dm"
    dest = TILE_CACHE / f"tile_{key}{ext}"

    if dest.exists():
        print(f"  cached  {dest.name}")
        return dest

    resp = requests.get(url, timeout=30, stream=True)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"  saved   {dest.name}  ({len(resp.content) / 1024:.1f} KB)")
    return dest


downloaded: list[pathlib.Path] = []
for url in glb_urls:
    try:
        p = download_tile(url)
        downloaded.append(p)
    except Exception as exc:
        print(f"  error: {exc}")

print(f"\n{len(downloaded)} tile(s) in {TILE_CACHE}/")

### 1.5 — Inspect a tile with pygltflib

`.b3dm` (Batched 3D Model) is just a 28-byte header prepended to a `.glb` payload.  
We strip that header before handing it to pygltflib.

In [ ]:
def load_glb_from_tile(path: pathlib.Path) -> pygltflib.GLTF2:
    """Load a .glb or .b3dm tile as a pygltflib GLTF2 object."""
    data = path.read_bytes()

    if path.suffix == ".b3dm":
        # b3dm layout:
        #   magic        4 bytes  "b3dm"
        #   version      4 bytes  uint32
        #   byteLength   4 bytes  uint32
        #   featureTableJSONByteLength  4 bytes
        #   featureTableBinaryByteLength 4 bytes
        #   batchTableJSONByteLength    4 bytes
        #   batchTableBinaryByteLength  4 bytes  ← total header = 28 bytes
        header = struct.unpack_from("<4sIIIIII", data)
        ft_json_len, ft_bin_len, bt_json_len, bt_bin_len = header[3:7]
        glb_start = 28 + ft_json_len + ft_bin_len + bt_json_len + bt_bin_len
        data = data[glb_start:]

    # Write to a temp file because pygltflib.load() needs a path
    tmp = path.with_suffix(".tmp.glb")
    tmp.write_bytes(data)
    gltf = pygltflib.GLTF2().load(str(tmp))
    tmp.unlink()
    return gltf


if downloaded:
    sample_path = downloaded[0]
    gltf = load_glb_from_tile(sample_path)

    print(f"File      : {sample_path.name}  ({sample_path.stat().st_size / 1024:.1f} KB)")
    print(f"Meshes    : {len(gltf.meshes)}")
    print(f"Materials : {len(gltf.materials)}")
    print(f"Textures  : {len(gltf.textures)}")
    print(f"Buffers   : {len(gltf.buffers)} — total {sum(b.byteLength for b in gltf.buffers) / 1024:.1f} KB")
    print(f"Extensions used: {gltf.extensionsUsed}")
else:
    print("No tiles downloaded — skipping inspection.")

---
## Part 2 — Meshopt Compression

### What `EXT_meshopt_compression` does

Standard GLB stores vertex buffers as raw binary.  
Meshopt encodes them with:

| Encoder | Best for | Typical saving |
|---|---|---|
| `meshopt_encodeVertexBuffer` | positions, normals, UVs | ~50–65% |
| `meshopt_encodeIndexBuffer` | index arrays | ~65–75% |

The data stays lossless. The decoder (`MeshoptDecoder.js` used in Three.js) runs in
a WASM module and is extremely fast.

### What the extension looks like in the JSON

```json
"bufferViews": [{
    "buffer": 0,
    "byteOffset": 0,
    "byteLength": 1024,
    "extensions": {
        "EXT_meshopt_compression": {
            "buffer": 1,          ← points at the compressed buffer
            "byteOffset": 0,
            "byteLength": 512,    ← smaller!
            "byteStride": 12,
            "mode": "ATTRIBUTES", ← or TRIANGLES / INDICES
            "count": 256
        }
    }
}]
```

### 2.1 — Compress vertex buffers with meshoptimizer

In [ ]:
import meshoptimizer as mo


def compress_glb_meshopt(src_path: pathlib.Path) -> pathlib.Path:
    """Re-encode every vertex/index buffer view in a GLB with meshopt.

    Returns the path to the compressed output file.
    """
    gltf = pygltflib.GLTF2().load(str(src_path))

    # Pull the raw binary blob out of the single GLB buffer
    blob: bytes = gltf.binary_blob()
    compressed_chunks: list[bytes] = []
    new_blob_offset = 0

    for bv in gltf.bufferViews:
        raw = blob[bv.byteOffset : bv.byteOffset + bv.byteLength]

        # Determine stride — accessors that point here tell us the component layout
        accessors_here = [a for a in gltf.accessors if a.bufferView == gltf.bufferViews.index(bv)]

        if bv.target == pygltflib.ELEMENT_ARRAY_BUFFER:
            # Index buffer: encode as triangle indices (stride = per-index byte width)
            index_stride = 2 if accessors_here and accessors_here[0].componentType == pygltflib.UNSIGNED_SHORT else 4
            index_count = bv.byteLength // index_stride
            encoded = mo.meshopt_encode_index_buffer(
                np.frombuffer(raw, dtype=np.uint16 if index_stride == 2 else np.uint32),
                index_count,
            )
            mode = "TRIANGLES"
            stride = index_stride
            count = index_count

        elif bv.target == pygltflib.ARRAY_BUFFER and accessors_here:
            # Vertex attribute buffer
            stride = bv.byteStride or (bv.byteLength // accessors_here[0].count)
            count = bv.byteLength // stride
            encoded = mo.meshopt_encode_vertex_buffer(
                np.frombuffer(raw, dtype=np.uint8),
                count,
                stride,
            )
            mode = "ATTRIBUTES"

        else:
            # Non-geometry buffer view (e.g. texture image) — keep as-is
            compressed_chunks.append(raw)
            new_blob_offset += len(raw)
            continue

        # Pad encoded chunk to 4-byte alignment (GLB requirement)
        padded = encoded + b"\x00" * ((-len(encoded)) % 4)

        # Attach extension to this bufferView
        if bv.extensions is None:
            bv.extensions = {}
        bv.extensions["EXT_meshopt_compression"] = {
            "buffer": 1,                     # compressed data lives in buffer #1
            "byteOffset": new_blob_offset,
            "byteLength": len(padded),
            "byteStride": stride,
            "mode": mode,
            "count": count,
        }

        compressed_chunks.append(padded)
        new_blob_offset += len(padded)

    compressed_blob = b"".join(compressed_chunks)

    # Add a second buffer holding all compressed data
    gltf.buffers.append(pygltflib.Buffer(byteLength=len(compressed_blob)))

    # Register the extension
    if "EXT_meshopt_compression" not in (gltf.extensionsUsed or []):
        gltf.extensionsUsed = list(gltf.extensionsUsed or []) + ["EXT_meshopt_compression"]

    # Save
    dest = src_path.with_stem(src_path.stem + "_meshopt")
    gltf.set_binary_blob(blob + compressed_blob)  # keep original + append compressed
    gltf.save(str(dest))
    return dest


print("compress_glb_meshopt() defined — ready.")

### 2.2 — Run compression on a downloaded tile (or any GLB)

In [ ]:
# Use a downloaded tile, or point at the project's own GLB:
# src_glb = pathlib.Path("web/frontend/public/assets/models/Google_LasVegas_Export_v32.glb")

if downloaded:
    # Filter to plain .glb files — b3dm needs the strip step first
    glb_files = [p for p in downloaded if p.suffix == ".glb"]
    if not glb_files:
        # Convert the first b3dm to a tmp glb
        p = downloaded[0]
        tmp_glb = TILE_CACHE / (p.stem + "_stripped.glb")
        gltf_obj = load_glb_from_tile(p)
        gltf_obj.save(str(tmp_glb))
        glb_files = [tmp_glb]

    src_glb = glb_files[0]

    original_size = src_glb.stat().st_size
    print(f"Original : {src_glb.name}  →  {original_size / 1024:.1f} KB")

    out_glb = compress_glb_meshopt(src_glb)
    compressed_size = out_glb.stat().st_size

    saving_pct = (1 - compressed_size / original_size) * 100
    print(f"Compressed: {out_glb.name}  →  {compressed_size / 1024:.1f} KB  (−{saving_pct:.1f}%)")
else:
    print("No tiles downloaded — point src_glb at any .glb file and run compress_glb_meshopt().")

### 2.3 — Verify: decode the compressed buffer and compare to original

In [ ]:
def verify_meshopt_roundtrip(original_path: pathlib.Path, compressed_path: pathlib.Path) -> None:
    """Decode the first compressed vertex buffer and compare to the original."""
    orig = pygltflib.GLTF2().load(str(original_path))
    comp = pygltflib.GLTF2().load(str(compressed_path))

    orig_blob = orig.binary_blob()
    comp_blob = comp.binary_blob()

    for i, (bv_orig, bv_comp) in enumerate(zip(orig.bufferViews, comp.bufferViews)):
        ext = (bv_comp.extensions or {}).get("EXT_meshopt_compression")
        if ext is None:
            continue

        # Decode
        enc_start = ext["byteOffset"]
        enc_end   = enc_start + ext["byteLength"]
        encoded   = np.frombuffer(comp_blob[enc_start:enc_end], dtype=np.uint8)

        mode   = ext["mode"]
        stride = ext["byteStride"]
        count  = ext["count"]

        if mode == "TRIANGLES":
            dtype = np.uint16 if stride == 2 else np.uint32
            decoded = mo.meshopt_decode_index_buffer(encoded, count, stride)
        else:
            decoded = mo.meshopt_decode_vertex_buffer(encoded, count, stride)

        # Compare to original bytes
        original_bytes = orig_blob[bv_orig.byteOffset : bv_orig.byteOffset + bv_orig.byteLength]
        decoded_bytes  = decoded.tobytes()[:bv_orig.byteLength]

        match = original_bytes == decoded_bytes
        print(f"  bufferView[{i}] mode={mode:12s}  {bv_orig.byteLength:6d}B → {ext['byteLength']:6d}B  "
              f"{'✓ lossless' if match else '✗ mismatch'}")
        break  # just check the first compressed view for demo purposes


if downloaded and 'out_glb' in dir() and out_glb.exists():
    verify_meshopt_roundtrip(src_glb, out_glb)
else:
    print("Skipped — run the compression cell first.")

### 2.4 — Size comparison across all tiles

In [ ]:
import io
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

results = []

for p in TILE_CACHE.glob("*.glb"):
    if "_meshopt" in p.stem or "_stripped" in p.stem:
        continue
    compressed = p.with_stem(p.stem + "_meshopt")
    if not compressed.exists():
        try:
            compressed = compress_glb_meshopt(p)
        except Exception as exc:
            print(f"  skip {p.name}: {exc}")
            continue
    results.append({
        "name": p.stem[:20],
        "original_kb": p.stat().st_size / 1024,
        "compressed_kb": compressed.stat().st_size / 1024,
    })

if results:
    print(f"{'Tile':<22} {'Original KB':>12} {'Meshopt KB':>12} {'Saving':>8}")
    print("-" * 58)
    for r in results:
        pct = (1 - r['compressed_kb'] / r['original_kb']) * 100
        print(f"{r['name']:<22} {r['original_kb']:>12.1f} {r['compressed_kb']:>12.1f} {pct:>7.1f}%")

    if HAS_MPL:
        names = [r["name"] for r in results]
        x = np.arange(len(names))
        fig, ax = plt.subplots(figsize=(max(6, len(names) * 1.5), 4))
        ax.bar(x - 0.2, [r["original_kb"] for r in results], 0.4, label="Original", color="#4e79a7")
        ax.bar(x + 0.2, [r["compressed_kb"] for r in results], 0.4, label="Meshopt", color="#f28e2b")
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=30, ha="right")
        ax.set_ylabel("KB")
        ax.set_title("GLB size: original vs EXT_meshopt_compression")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No tiles to compare yet.")

---
## Summary — the full pipeline

```
Google Maps (Chrome)
      │  GPU draw calls
      ▼
RenderDoc  (.rdc capture)
      │  MapsModelsImporter Blender add-on
      ▼
Blender scene  (32 captures stitched with LilyCaptureMerger)
      │  File > Export > glTF 2.0 (.glb)
      ▼                with Meshopt enabled (same as compress_glb_meshopt() above)
Google_LasVegas_Export_v32.glb
      │  GLTFLoader + MeshoptDecoder
      ▼
Three.js scene  (web/frontend)
```

**Alternatively** — if you have an API key and want a fully-programmable pipeline:

```
Part 1 cells above   →  raw tile GLBs
      │  stitch + merge in Blender or Open3D
      ▼
Part 2 cells above   →  meshopt-compressed GLB
      │  drop into web/frontend/public/assets/models/
      ▼
map3d.html           →  instant 3D city in the browser
```